In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    TOY STABLE DIFFUSION TRAINING PIPELINE                    ║
║                        Cell 1: Environment Setup & Imports                   ║
╚══════════════════════════════════════════════════════════════════════════════╝

A production-ready training pipeline for building a Stable Diffusion model from
scratch on a single T4 GPU (16GB VRAM).

Author: Your Learning Journey
Target: 128x128 image generation with ~86M param U-Net
Dataset: MS COCO (118K images with captions)
"""

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 1: MOUNT GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════════════════

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define base paths
PROJECT_ROOT = "/content/drive/MyDrive/ToyStableDiffusion"
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
LOGS_DIR = os.path.join(PROJECT_ROOT, "logs")
SAMPLES_DIR = os.path.join(PROJECT_ROOT, "samples")
DATA_DIR = "/content/data"  # Local to Colab for faster I/O

# Create directories
for dir_path in [PROJECT_ROOT, CHECKPOINTS_DIR, LOGS_DIR, SAMPLES_DIR, DATA_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print(f"✓ Google Drive mounted successfully")
print(f"✓ Project root: {PROJECT_ROOT}")

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 2: INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════════════════

!pip install torch torchvision torchaudio
!pip install transformers datasets
!pip install bitsandbytes  # For 8-bit optimizers
!pip install accelerate
!pip install wandb  # For experiment tracking
!pip install einops  # For tensor operations
!pip install pillow numpy matplotlib tqdm
!pip install torchinfo  # For model summary
!pip install ftfy regex  # For text processing

print("✓ All dependencies installed")

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 3: IMPORTS
# ═══════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
from torchvision.utils import save_image, make_grid

import bitsandbytes as bnb
from transformers import get_cosine_schedule_with_warmup
from datasets get load_dataset

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import json
import time
from datetime import datetime
from typing import Optional, Tuple, List, Dict, Any
from dataclasses import dataclass, asdict
from pathlib import Path
from tqdm.auto import tqdm
import math
import random
import gc
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 4: CONFIGURATION CLASSES
# ═══════════════════════════════════════════════════════════════════════════

@dataclass
class VAEConfig:
    """Configuration for Variational Autoencoder"""
    image_size: int = 128
    latent_size: int = 16  # 128 -> 16 (8x compression)
    in_channels: int = 3
    latent_channels: int = 4
    hidden_dims: List[int] = None
    beta: float = 1.0  # KL divergence weight

    def __post_init__(self):
        if self.hidden_dims is None:
            # Progressive channel expansion
            self.hidden_dims = [64, 128, 256, 512]

@dataclass
class CLIPConfig:
    """Configuration for CLIP Text Encoder"""
    vocab_size: int = 10000  # Toy vocabulary
    embed_dim: int = 512
    num_layers: int = 6
    num_heads: int = 8
    mlp_ratio: int = 4
    max_seq_length: int = 77
    dropout: float = 0.1

    # Image encoder config
    image_size: int = 128
    patch_size: int = 16
    vision_layers: int = 6

@dataclass
class UNetConfig:
    """Configuration for Diffusion U-Net"""
    image_size: int = 16  # Latent space size
    in_channels: int = 4  # Latent channels
    out_channels: int = 4
    model_channels: int = 192  # Base channel count
    num_res_blocks: int = 2
    attention_resolutions: Tuple[int] = (8, 4, 2)
    channel_mult: Tuple[int] = (1, 2, 3, 4)
    dropout: float = 0.1
    num_heads: int = 8
    context_dim: int = 512  # CLIP embedding dim
    use_checkpoint: bool = False  # Gradient checkpointing

@dataclass
class TrainingConfig:
    """Master training configuration"""
    # General
    component: str = "vae"  # "vae", "clip", or "unet"
    batch_size: int = 16
    num_epochs: int = 100
    learning_rate: float = 1e-6
    weight_decay: float = 0.01
    warmup_steps: int = 1000

    # Optimization
    use_8bit_adam: bool = True
    gradient_clip: float = 1.0
    use_amp: bool = False  # Automatic Mixed Precision

    # Checkpointing & Logging
    save_every: int = 5000
    sample_every: int = 2000
    log_every: int = 100
    num_samples: int = 4

    # EMA (Exponential Moving Average)
    use_ema: bool = True
    ema_decay: float = 0.9999

    # Diffusion specific
    num_diffusion_steps: int = 1000
    cfg_dropout: float = 0.1  # Classifier-free guidance dropout

    # Paths
    checkpoint_path: Optional[str] = None
    vae_path: Optional[str] = None
    clip_path: Optional[str] = None

# Global configs
vae_config = VAEConfig()
clip_config = CLIPConfig()
unet_config = UNetConfig()
train_config = TrainingConfig()

print("✓ Configuration classes initialized")
print(f"  VAE params: ~{sum([np.prod(list(vae_config.hidden_dims)) for _ in range(2)])//1000}K (estimated)")
print(f"  Target U-Net params: ~86M")
print(f"  Training component: {train_config.component}")

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 5: UTILITY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

class Logger:
    """Comprehensive logging utility"""
    def __init__(self, log_dir: str, experiment_name: str):
        self.log_dir = Path(log_dir)
        self.experiment_name = experiment_name
        self.log_file = self.log_dir / f"{experiment_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        self.metrics = []

    def log(self, message: str, level: str = "INFO"):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_message = f"[{timestamp}] [{level}] {message}"
        print(log_message)
        with open(self.log_file, "a") as f:
            f.write(log_message + "\n")

    def log_metrics(self, step: int, metrics: Dict[str, float]):
        metrics["step"] = step
        metrics["timestamp"] = time.time()
        self.metrics.append(metrics)

        # Log to console
        metrics_str = " | ".join([f"{k}: {v:.6f}" if isinstance(v, float) else f"{k}: {v}"
                                  for k, v in metrics.items()])
        self.log(f"Step {step} - {metrics_str}")

        # Save metrics to JSON
        with open(self.log_dir / f"{self.experiment_name}_metrics.json", "w") as f:
            json.dump(self.metrics, f, indent=2)

    def plot_metrics(self, metric_names: List[str]):
        """Plot training curves"""
        fig, axes = plt.subplots(1, len(metric_names), figsize=(6*len(metric_names), 4))
        if len(metric_names) == 1:
            axes = [axes]

        for ax, metric_name in zip(axes, metric_names):
            steps = [m["step"] for m in self.metrics if metric_name in m]
            values = [m[metric_name] for m in self.metrics if metric_name in m]
            ax.plot(steps, values)
            ax.set_xlabel("Step")
            ax.set_ylabel(metric_name)
            ax.set_title(f"{metric_name} over training")
            ax.grid(True)

        plt.tight_layout()
        plt.savefig(self.log_dir / f"{self.experiment_name}_curves.png", dpi=150)
        plt.close()

def count_parameters(model: nn.Module) -> int:
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def format_time(seconds: float) -> str:
    """Format seconds into human-readable time"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: Any,
    epoch: int,
    step: int,
    loss: float,
    path: str,
    ema_model: Optional[nn.Module] = None,
    scaler: Optional[GradScaler] = None
):
    """Save comprehensive checkpoint"""
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "epoch": epoch,
        "step": step,
        "loss": loss,
        "scaler_state_dict": scaler.state_dict() if scaler else None,
    }

    if ema_model is not None:
        checkpoint["ema_model_state_dict"] = ema_model.state_dict()

    torch.save(checkpoint, path)
    print(f"✓ Checkpoint saved: {path}")

def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[Any] = None,
    ema_model: Optional[nn.Module] = None,
    scaler: Optional[GradScaler] = None
) -> Tuple[int, int, float]:
    """Load checkpoint and return epoch, step, loss"""
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler and checkpoint["scheduler_state_dict"]:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if ema_model and "ema_model_state_dict" in checkpoint:
        ema_model.load_state_dict(checkpoint["ema_model_state_dict"])

    if scaler and checkpoint.get("scaler_state_dict"):
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    epoch = checkpoint.get("epoch", 0)
    step = checkpoint.get("step", 0)
    loss = checkpoint.get("loss", float('inf'))

    print(f"✓ Checkpoint loaded: {path}")
    print(f"  Resuming from epoch {epoch}, step {step}, loss {loss:.6f}")

    return epoch, step, loss

class EMA:
    """Exponential Moving Average for model weights"""
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}

        # Initialize shadow weights
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        """Update EMA weights"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                assert name in self.shadow
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        """Apply EMA weights to model (for inference)"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        """Restore original weights"""
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

print("✓ Utility functions and Logger class defined")
print("\n" + "="*80)
print("SETUP COMPLETE - Ready to proceed to Cell 2 (Dataset Download)")
print("="*80)

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        Cell 2: Dataset Preparation (FAST .zip Copy Method)                   ║
╚══════════════════════════════════════════════════════════════════════════════╝

This cell copies the pre-zipped coco_128.zip from Google Drive to the fast
Colab disk and unpacks it. This is the fastest method (3-4 mins total).

ASSUMES YOU HAVE MANUALLY CREATED AND UPLOADED "coco_128.zip" TO YOUR DRIVE.
"""

import os
import shutil
from PIL import Image
import json
from tqdm.auto import tqdm
import re
from collections import Counter, defaultdict
import pandas as pd
import gc
import time
import zipfile

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 1: DATASET SETUP (This runs every time)
# ═══════════════════════════════════════════════════════════════════════════

# --- Use the paths from Cell 1 ---
# PERMANENT dataset storage on Google Drive
DRIVE_ZIP_PATH = os.path.join(PROJECT_ROOT, "coco_128.zip")
# FAST, TEMPORARY storage on Colab's local disk
LOCAL_DATA_DIR_COCO = os.path.join(DATA_DIR, "coco_2017_dataset_128")
LOCAL_ZIP_PATH = os.path.join(DATA_DIR, "coco_128.zip")

# This is what the rest of the script will use
DATASET_DIR = LOCAL_DATA_DIR_COCO
TOKENIZER_PATH = os.path.join(PROJECT_ROOT, "tokenizer.pt")

print("="*80)
print("Initializing Dataset...")
print("="*80)

# --- 1. Verify Google Drive Connection ---
print("Verifying Google Drive connection...")
try:
    os.listdir(PROJECT_ROOT)
    time.sleep(2) # Give Drive a moment to sync
    print("✓ Google Drive connection verified.")
except Exception as e:
    print(f"❌ CRITICAL ERROR: Cannot access project root: {PROJECT_ROOT}")
    raise e

# --- 2. Check for the .zip file on Drive ---
if not os.path.exists(DRIVE_ZIP_PATH):
    print(f"❌ ERROR: Dataset archive not found at {DRIVE_ZIP_PATH}")
    print("   Please manually create 'coco_128.zip' on your local PC")
    print(f"   and upload it to: {PROJECT_ROOT}")
    raise FileNotFoundError(f"Archive not found: {DRIVE_ZIP_PATH}")
else:
    print(f"✓ Found dataset archive on Google Drive.")

# --- 3. Copy and Unpack (if not already done) ---
if not os.path.exists(LOCAL_DATA_DIR_COCO):
    print(f"\nCopying dataset archive from Google Drive to fast Colab disk...")
    print(f"FROM: {DRIVE_ZIP_PATH}")
    print(f"TO:   {LOCAL_ZIP_PATH}")
    print("⏳ This may take 1-2 minutes...")

    start_time = time.time()
    shutil.copyfile(DRIVE_ZIP_PATH, LOCAL_ZIP_PATH)
    print(f"✓ Copy complete! (Time: {time.time() - start_time:.2f}s)")

    print(f"\nUnpacking archive on fast Colab disk...")
    print(f"TO: {LOCAL_DATA_DIR_COCO}")
    print("⏳ This may take 1-2 minutes...")

    start_time = time.time()
    # We must create the target directory FIRST
    os.makedirs(LOCAL_DATA_DIR_COCO, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR_COCO)

    print(f"✓ Unpack complete! (Time: {time.time() - start_time:.2f}s)")

    # --- CHECK FOR NESTED FOLDER ---
    # This checks if you zipped the folder *itself* (e.g., coco_2017_dataset_128)
    # If so, it "moves" everything up one level to fix it.
    nested_dir = os.path.join(LOCAL_DATA_DIR_COCO, "COCO_PROCESSED_128")
    if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
        print("  Fixing nested zip folder (moving files up one level)...")
        for item in os.listdir(nested_dir):
            shutil.move(os.path.join(nested_dir, item), LOCAL_DATA_DIR_COCO)
        os.rmdir(nested_dir)
        print("  ✓ Nested folder fixed.")

    # Clean up the local .zip file to save space
    os.remove(LOCAL_ZIP_PATH)
    print("✓ Cleaned up local .zip file.")
else:
    print(f"\n✓ Dataset already exists on fast Colab disk.")


# ═══════════════════════════════════════════════════════════════════════════
# SECTION 2: DATASET CLASSES (Reads from FAST local disk)
# ═══════════════════════════════════════════════════════════════════════════

class ImageCaptionDataset(Dataset):
    """Dataset for image-text pairs (reads from local Colab disk)"""

    def __init__(
        self,
        data_dir: str = LOCAL_DATA_DIR_COCO, # Default to local dir
        transform=None,
        mode: str = "train",
        val_split: float = 0.05
    ):
        self.data_dir = data_dir
        self.images_dir = os.path.join(data_dir, "images")
        self.annotations_file = os.path.join(data_dir, "annotations.json")
        self.transform = transform
        self.mode = mode

        # Load annotations
        try:
            with open(self.annotations_file, 'r') as f:
                all_annotations = json.load(f)
        except FileNotFoundError:
            print(f"❌ CRITICAL: annotations.json not found at {self.annotations_file}")
            print("   Check if your zip file structure is correct.")
            raise

        # Split into train/val
        num_val = int(len(all_annotations) * val_split)
        if mode == "val":
            self.annotations = all_annotations[:num_val]
        else:
            self.annotations = all_annotations[num_val:]

        print(f"✓ {mode.upper()} dataset initialized: {len(self.annotations)} samples")

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        image_path = os.path.join(self.images_dir, ann["image_filename"])
        try:
            image = Image.open(image_path).convert("RGB")
        except FileNotFoundError:
            # Fallback for missing images (robustness)
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            image = self.transform(image)

        captions = ann["captions"]
        caption = random.choice(captions) if captions else ""

        return {"image": image, "caption": caption, "image_id": ann["image_id"]}

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 3: TRANSFORMS & DATA LOADERS
# ═══════════════════════════════════════════════════════════════════════════

def get_transforms(component: str):
    if component == "vae":
        return T.Compose([T.ToTensor(), T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
    elif component == "clip":
        return T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    elif component == "unet":
        return T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        ])
    else:
        raise ValueError(f"Unknown component: {component}")

def get_dataloader(component: str, batch_size: int, num_workers: int = 2):
    transform = get_transforms(component)
    train_dataset = ImageCaptionDataset(transform=transform, mode="train")
    val_dataset = ImageCaptionDataset(transform=transform, mode="val")
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True, drop_last=False
    )
    return train_loader, val_loader

print("\n" + "="*80)
print("Testing dataloader...")
print("="*80)

# Test dataloader
try:
    test_loader, _ = get_dataloader("vae", batch_size=4)
    test_batch = next(iter(test_loader))
    print(f"✓ Batch loaded successfully!")
    print(f"  Images shape: {test_batch['image'].shape}")
    print(f"  First caption: {test_batch['caption'][0][:80]}...")
except Exception as e:
    print(f"❌ Dataloader Test Failed: {e}")

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 4: TEXT TOKENIZER
# ═══════════════════════════════════════════════════════════════════════════

class SimpleTokenizer:
    def __init__(self, vocab_size: int = 10000, max_length: int = 77):
        self.vocab_size = vocab_size
        self.max_length = max_length
        self.word2idx = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.vocab_built = False

    def build_vocab(self, captions: list):
        word_freq = Counter()
        for caption in tqdm(captions, desc="Building vocab"):
            words = re.findall(r'\b\w+\b', caption.lower())
            word_freq.update(words)
        most_common = word_freq.most_common(self.vocab_size - len(self.word2idx))
        for word, _ in most_common:
            if word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
        self.vocab_built = True
        print(f"✓ Vocabulary built: {len(self.word2idx)} words")

    def encode(self, text: str) -> torch.Tensor:
        words = re.findall(r'\b\w+\b', text.lower())
        ids = [self.word2idx["<SOS>"]]
        for word in words[:self.max_length - 2]:
            ids.append(self.word2idx.get(word, self.word2idx["<UNK>"]))
        ids.append(self.word2idx["<EOS>"])
        ids += [self.word2idx["<PAD>"]] * (self.max_length - len(ids))
        return torch.tensor(ids[:self.max_length], dtype=torch.long)

    def decode(self, ids: torch.Tensor) -> str:
        words = []
        for idx in ids:
            idx = idx.item() if torch.is_tensor(idx) else idx
            word = self.idx2word.get(idx, "<UNK>")
            if word in ["<PAD>", "<SOS>", "<EOS>"]: continue
            words.append(word)
        return " ".join(words)

print("\n" + "="*80)
print("Building tokenizer...")
print("="*80)

tokenizer = SimpleTokenizer(vocab_size=clip_config.vocab_size, max_length=clip_config.max_seq_length)

# Check if tokenizer is saved on Drive
if not os.path.exists(TOKENIZER_PATH):
    print(f"Tokenizer not found on Drive. Building...")
    annotations_file = os.path.join(LOCAL_DATA_DIR_COCO, "annotations.json")
    if os.path.exists(annotations_file):
        with open(annotations_file, 'r') as f:
            annotations = json.load(f)
        all_captions = [cap for ann in annotations if ann['captions'] for cap in ann["captions"]]

        print(f"Building vocabulary from {len(all_captions)} captions...")
        tokenizer.build_vocab(all_captions)

        torch.save({
            "word2idx": tokenizer.word2idx, "idx2word": tokenizer.idx2word,
            "vocab_size": tokenizer.vocab_size, "max_length": tokenizer.max_length
        }, TOKENIZER_PATH)
        print(f"✓ Tokenizer saved: {TOKENIZER_PATH}")
    else:
        print("⚠️ Warning: annotations.json not found, cannot build tokenizer yet.")
else:
    print(f"✓ Loading tokenizer from {TOKENIZER_PATH}...")
    tokenizer_data = torch.load(TOKENIZER_PATH, map_location=device)
    tokenizer.word2idx = tokenizer_data["word2idx"]
    tokenizer.idx2word = tokenizer_data["idx2word"]
    tokenizer.vocab_built = True
    print(f"✓ Tokenizer loaded with {len(tokenizer.word2idx)} words.")

# ═══════════════════════════════════════════════════════════════════════════
# SECTION 5: VISUALIZE SAMPLES
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("Visualizing sample images...")
print("="*80)

try:
    sample_loader, _ = get_dataloader("vae", batch_size=8)
    sample_batch = next(iter(sample_loader))
    sample_images = (sample_batch['image'] + 1) / 2
    sample_captions = sample_batch['caption']
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for idx in range(8):
        img = sample_images[idx].permute(1, 2, 0).cpu().numpy()
        caption = sample_captions[idx][:50]
        axes[idx].imshow(img)
        axes[idx].set_title(caption, fontsize=8)
        axes[idx].axis('off')
    plt.suptitle("Sample Images from MS-COCO Dataset", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAMPLES_DIR, "dataset_samples.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Sample visualization saved to:", os.path.join(SAMPLES_DIR, "dataset_samples.png"))
except Exception as e:
    print(f"⚠️ Visualization skipped: {e}")

# ═══════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("DATASET PREPARATION COMPLETE")
print("="*80)
print(f"✓ Permanent data (Drive): {DRIVE_ZIP_PATH}")
print(f"✓ Local data (Colab):   {LOCAL_DATA_DIR_COCO}")
print(f"✓ Tokenizer (Drive):    {TOKENIZER_PATH}")
print("\n✅ Ready for training! Proceed to Cell 3 (Model Architectures)")
print("="*80)

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                      Cell 3: Model Architectures                             ║
╚══════════════════════════════════════════════════════════════════════════════╝

Defines all three model architectures:
1. VAE (Variational Autoencoder) - 128x128 -> 16x16 compression
2. CLIP (Text & Image Encoders) - Text-Image alignment
3. U-Net (Diffusion Model) - ~86M parameters
"""

from einops import rearrange
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

# ═══════════════════════════════════════════════════════════════════════════
# PART 1: VAE (VARIATIONAL AUTOENCODER)
# ═══════════════════════════════════════════════════════════════════════════

class ResidualBlock(nn.Module):
    """Residual block with GroupNorm"""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        # Use GroupNorm(8) only if channels are sufficient, else use Identity or smaller groups
        groups = 8
        if in_channels < 8: groups = 1 # Fallback for small channels

        self.norm1 = nn.GroupNorm(groups, in_channels)
        self.norm2 = nn.GroupNorm(8, out_channels) # Output is usually large enough
        self.act = nn.SiLU()

        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

class Encoder(nn.Module):
    """VAE Encoder: 128x128 -> 16x16"""
    def __init__(self, in_channels: int, hidden_dims: list, latent_channels: int):
        super().__init__()

        # --- FIX: Initial projection (RGB 3 -> Hidden Dim 64) ---
        # This ensures we never try to GroupNorm the 3-channel input
        self.initial_conv = nn.Conv2d(in_channels, hidden_dims[0], kernel_size=3, padding=1)

        layers = []
        prev_dim = hidden_dims[0]

        # Downsample: 128 -> 64 -> 32 -> 16
        for h_dim in hidden_dims:
            layers.append(ResidualBlock(prev_dim, h_dim))
            layers.append(nn.Conv2d(h_dim, h_dim, 3, stride=2, padding=1))  # Downsample
            prev_dim = h_dim

        self.encoder = nn.Sequential(*layers)

        # Output layer (mean and logvar)
        self.fc_mu = nn.Conv2d(hidden_dims[-1], latent_channels, 1)
        self.fc_logvar = nn.Conv2d(hidden_dims[-1], latent_channels, 1)

    def forward(self, x):
        x = self.initial_conv(x) # <--- Apply projection first
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

class Decoder(nn.Module):
    """VAE Decoder: 16x16 -> 128x128"""
    def __init__(self, latent_channels: int, hidden_dims: list, out_channels: int):
        super().__init__()

        self.initial_conv = nn.Conv2d(latent_channels, hidden_dims[-1], 3, padding=1)

        hidden_dims = list(reversed(hidden_dims))
        layers = []
        prev_dim = hidden_dims[0]

        # Upsample: 16 -> 32 -> 64 -> 128
        for h_dim in hidden_dims:
            layers.append(ResidualBlock(prev_dim, h_dim))
            layers.append(nn.ConvTranspose2d(h_dim, h_dim, 4, stride=2, padding=1))  # Upsample
            prev_dim = h_dim

        self.decoder = nn.Sequential(*layers)

        # Output layer
        self.out = nn.Sequential(
            nn.GroupNorm(8, hidden_dims[-1]),
            nn.SiLU(),
            nn.Conv2d(hidden_dims[-1], out_channels, 3, padding=1)
        )

    def forward(self, z):
        z = self.initial_conv(z)
        h = self.decoder(z)
        return self.out(h)

class VAE(nn.Module):
    """Complete VAE: 128x128 <-> 16x16 latent space"""
    def __init__(self, config: VAEConfig):
        super().__init__()
        self.config = config

        self.encoder = Encoder(
            config.in_channels,
            config.hidden_dims,
            config.latent_channels
        )

        self.decoder = Decoder(
            config.latent_channels,
            config.hidden_dims,
            config.in_channels
        )

        # Free bits technique to prevent collapse
        self.free_bits = 0.5  # Minimum KL per latent dimension

    def reparameterize(self, mu, logvar):
        """Reparameterization trick with numerical stability"""
        # Clamp logvar BEFORE exp()
        logvar = torch.clamp(logvar, -10, 10)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, x):
        """Encode to latent space"""
        mu, logvar = self.encoder(x)

        # CRITICAL: Clamp both mu and logvar
        mu = torch.clamp(mu, -10, 10)
        logvar = torch.clamp(logvar, -10, 10)

        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def decode(self, z):
        """Decode from latent space"""
        return self.decoder(z)

    def forward(self, x):
        z, mu, logvar = self.encode(x)
        recon = self.decode(z)
        return recon, mu, logvar

# ═══════════════════════════════════════════════════════════════════════════
# PART 2: CLIP (TEXT & IMAGE ENCODERS)
# ═══════════════════════════════════════════════════════════════════════════

class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism"""
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert self.head_dim * num_heads == embed_dim

        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, N, C = x.shape

        # Compute Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)

        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # Combine heads
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)

        return x

class TransformerBlock(nn.Module):
    """Transformer block"""
    def __init__(self, embed_dim: int, num_heads: int, mlp_ratio: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

        mlp_dim = embed_dim * mlp_ratio
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.mlp(self.norm2(x))
        return x

class TextEncoder(nn.Module):
    """CLIP Text Encoder"""
    def __init__(self, config: CLIPConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, config.max_seq_length, config.embed_dim))

        self.transformer = nn.ModuleList([
            TransformerBlock(config.embed_dim, config.num_heads, config.mlp_ratio, config.dropout)
            for _ in range(config.num_layers)
        ])

        self.norm = nn.LayerNorm(config.embed_dim)

    def forward(self, text_ids):
        """
        Args:
            text_ids: (B, seq_len)
        Returns:
            text_features: (B, embed_dim)
        """
        B, seq_len = text_ids.shape

        # Embeddings
        x = self.token_embedding(text_ids)
        x = x + self.pos_embedding[:, :seq_len, :]

        # Transformer
        for block in self.transformer:
            x = block(x)

        x = self.norm(x)

        # Take the [EOS] token embedding (last non-padding token)
        # For simplicity, we'll take the first token
        text_features = x[:, 0, :]

        # Normalize
        text_features = F.normalize(text_features, dim=-1)

        return text_features

class ImageEncoder(nn.Module):
    """CLIP Image Encoder (Vision Transformer)"""
    def __init__(self, config: CLIPConfig):
        super().__init__()
        self.config = config

        # Patch embedding
        self.patch_size = config.patch_size
        self.num_patches = (config.image_size // config.patch_size) ** 2
        patch_dim = 3 * config.patch_size * config.patch_size

        self.patch_embed = nn.Linear(patch_dim, config.embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, config.embed_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, config.embed_dim))

        self.transformer = nn.ModuleList([
            TransformerBlock(config.embed_dim, config.num_heads, config.mlp_ratio, config.dropout)
            for _ in range(config.vision_layers)
        ])

        self.norm = nn.LayerNorm(config.embed_dim)

    def forward(self, images):
        """
        Args:
            images: (B, 3, H, W)
        Returns:
            image_features: (B, embed_dim)
        """
        B = images.shape[0]

        # Patchify
        x = rearrange(images, 'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
                     p1=self.patch_size, p2=self.patch_size)

        # Patch embedding
        x = self.patch_embed(x)

        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Add position embedding
        x = x + self.pos_embedding

        # Transformer
        for block in self.transformer:
            x = block(x)

        x = self.norm(x)

        # Take CLS token
        image_features = x[:, 0, :]

        # Normalize
        image_features = F.normalize(image_features, dim=-1)

        return image_features

class CLIP(nn.Module):
    """Complete CLIP model"""
    def __init__(self, config: CLIPConfig):
        super().__init__()
        self.config = config

        self.text_encoder = TextEncoder(config)
        self.image_encoder = ImageEncoder(config)

        # Temperature parameter for contrastive learning
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, images, text_ids):
        """
        Args:
            images: (B, 3, H, W)
            text_ids: (B, seq_len)
        Returns:
            logits_per_image: (B, B)
            logits_per_text: (B, B)
        """
        image_features = self.image_encoder(images)
        text_features = self.text_encoder(text_ids)

        # Compute similarity
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()

        return logits_per_image, logits_per_text

# ═══════════════════════════════════════════════════════════════════════════
# PART 3: U-NET (DIFFUSION MODEL) - ~86M PARAMETERS
# ═══════════════════════════════════════════════════════════════════════════

class TimestepEmbedding(nn.Module):
    """Sinusoidal timestep embedding"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps):
        device = timesteps.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = timesteps[:, None] * embeddings[None, :]
        embeddings = torch.cat([torch.sin(embeddings), torch.cos(embeddings)], dim=-1)
        return embeddings

class ResBlock(nn.Module):
    """Residual block with time and context conditioning"""
    def __init__(self, in_channels: int, out_channels: int, time_emb_dim: int,
                 dropout: float = 0.1, use_checkpoint: bool = False):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        self.norm1 = nn.GroupNorm(32, in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)

        self.time_emb = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )

        self.norm2 = nn.GroupNorm(32, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        self.act = nn.SiLU()

    def _forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)

        # Add time embedding
        h = h + self.time_emb(t_emb)[:, :, None, None]

        h = self.act(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)

        return h + self.skip(x)

    def forward(self, x, t_emb):
        if self.use_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward, x, t_emb)
        else:
            return self._forward(x, t_emb)

class SpatialTransformer(nn.Module):
    """Spatial transformer for cross-attention with text"""
    def __init__(self, channels: int, context_dim: int, num_heads: int = 8):
        super().__init__()
        self.norm = nn.GroupNorm(32, channels)
        self.proj_in = nn.Conv2d(channels, channels, 1)

        self.transformer_blocks = nn.ModuleList([
            CrossAttentionBlock(channels, context_dim, num_heads)
        ])

        self.proj_out = nn.Conv2d(channels, channels, 1)

    def forward(self, x, context):
        B, C, H, W = x.shape
        x_in = x

        x = self.norm(x)
        x = self.proj_in(x)

        # Reshape for attention
        x = rearrange(x, 'b c h w -> b (h w) c')

        for block in self.transformer_blocks:
            x = block(x, context)

        # Reshape back
        x = rearrange(x, 'b (h w) c -> b c h w', h=H, w=W)
        x = self.proj_out(x)

        return x + x_in

class CrossAttentionBlock(nn.Module):
    """Cross-attention block"""
    def __init__(self, dim: int, context_dim: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn1 = MultiHeadAttention(dim, num_heads)  # Self-attention

        self.norm2 = nn.LayerNorm(dim)
        self.attn2 = CrossAttention(dim, context_dim, num_heads)  # Cross-attention

        self.norm3 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, x, context):
        x = x + self.attn1(self.norm1(x))
        x = x + self.attn2(self.norm2(x), context)
        x = x + self.mlp(self.norm3(x))
        return x

class CrossAttention(nn.Module):
    """Cross-attention mechanism"""
    def __init__(self, query_dim: int, context_dim: int, num_heads: int):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = query_dim // num_heads

        self.to_q = nn.Linear(query_dim, query_dim)
        self.to_k = nn.Linear(context_dim, query_dim)
        self.to_v = nn.Linear(context_dim, query_dim)
        self.to_out = nn.Linear(query_dim, query_dim)

    def forward(self, x, context):
        B, N, C = x.shape

        q = self.to_q(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        k = self.to_k(context).reshape(B, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        v = self.to_v(context).reshape(B, -1, self.num_heads, self.head_dim).permute(0, 2, 1, 3)

        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)

        out = (attn @ v).permute(0, 2, 1, 3).reshape(B, N, C)
        out = self.to_out(out)

        return out

class UNet(nn.Module):
    """Diffusion U-Net with cross-attention - ~86M parameters"""
    def __init__(self, config: UNetConfig):
        super().__init__()
        self.config = config

        # Time embedding
        time_emb_dim = config.model_channels * 4
        self.time_embed = nn.Sequential(
            TimestepEmbedding(config.model_channels),
            nn.Linear(config.model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )

        # Initial convolution
        self.conv_in = nn.Conv2d(config.in_channels, config.model_channels, 3, padding=1)

        # Downsampling
        self.down_blocks = nn.ModuleList([])
        channels = [config.model_channels]
        now_channels = config.model_channels

        for level, mult in enumerate(config.channel_mult):
            out_channels = config.model_channels * mult

            for _ in range(config.num_res_blocks):
                self.down_blocks.append(nn.ModuleList([
                    ResBlock(now_channels, out_channels, time_emb_dim,
                            config.dropout, config.use_checkpoint),
                    SpatialTransformer(out_channels, config.context_dim, config.num_heads)
                    if config.image_size // (2 ** level) in config.attention_resolutions else None
                ]))
                now_channels = out_channels
                channels.append(now_channels)

            # Downsample
            if level != len(config.channel_mult) - 1:
                self.down_blocks.append(nn.ModuleList([
                    nn.Conv2d(now_channels, now_channels, 3, stride=2, padding=1),
                    None
                ]))
                channels.append(now_channels)

        # Middle
        self.mid_block1 = ResBlock(now_channels, now_channels, time_emb_dim,
                                   config.dropout, config.use_checkpoint)
        self.mid_attn = SpatialTransformer(now_channels, config.context_dim, config.num_heads)
        self.mid_block2 = ResBlock(now_channels, now_channels, time_emb_dim,
                                   config.dropout, config.use_checkpoint)

        # Upsampling
        self.up_blocks = nn.ModuleList([])

        for level, mult in enumerate(reversed(config.channel_mult)):
            out_channels = config.model_channels * mult

            for i in range(config.num_res_blocks + 1):
                self.up_blocks.append(nn.ModuleList([
                    ResBlock(now_channels + channels.pop(), out_channels, time_emb_dim,
                            config.dropout, config.use_checkpoint),
                    SpatialTransformer(out_channels, config.context_dim, config.num_heads)
                    if config.image_size // (2 ** (len(config.channel_mult) - 1 - level)) in config.attention_resolutions else None
                ]))
                now_channels = out_channels

            # Upsample
            if level != len(config.channel_mult) - 1:
                self.up_blocks.append(nn.ModuleList([
                    nn.ConvTranspose2d(now_channels, now_channels, 4, stride=2, padding=1),
                    None
                ]))

        # Output
        self.out = nn.Sequential(
            nn.GroupNorm(32, now_channels),
            nn.SiLU(),
            nn.Conv2d(now_channels, config.out_channels, 3, padding=1)
        )

    def forward(self, x, timesteps, context):
        """
        Args:
            x: (B, C, H, W) - noisy latent
            timesteps: (B,) - timestep
            context: (B, seq_len, context_dim) - text embedding
        Returns:
            noise_pred: (B, C, H, W)
        """
        # Time embedding
        t_emb = self.time_embed(timesteps)

        # Initial conv
        h = self.conv_in(x)

        # Store skip connections
        skips = [h]

        # Downsample
        for block, attn in self.down_blocks:
            if isinstance(block, nn.Conv2d):
                h = block(h)
            else:
                h = block(h, t_emb)
                if attn is not None:
                    h = attn(h, context)
            skips.append(h)

        # Middle
        h = self.mid_block1(h, t_emb)
        h = self.mid_attn(h, context)
        h = self.mid_block2(h, t_emb)

        # Upsample
        for block, attn in self.up_blocks:
            if isinstance(block, nn.ConvTranspose2d):
                h = block(h)
            else:
                h = torch.cat([h, skips.pop()], dim=1)
                h = block(h, t_emb)
                if attn is not None:
                    h = attn(h, context)

        # Output
        return self.out(h)

# ═══════════════════════════════════════════════════════════════════════════
# MODEL INSTANTIATION & PARAMETER COUNT
# ═══════════════════════════════════════════════════════════════════════════

print("\n" + "="*80)
print("INITIALIZING MODELS")
print("="*80)

# Create models
vae_model = VAE(vae_config).to(device)
clip_model = CLIP(clip_config).to(device)
unet_model = UNet(unet_config).to(device)

# Count parameters
vae_params = count_parameters(vae_model)
clip_params = count_parameters(clip_model)
unet_params = count_parameters(unet_model)

print(f"\n✓ VAE initialized")
print(f"  Parameters: {vae_params:,} ({vae_params/1e6:.2f}M)")

print(f"\n✓ CLIP initialized")
print(f"  Text Encoder: {count_parameters(clip_model.text_encoder):,}")
print(f"  Image Encoder: {count_parameters(clip_model.image_encoder):,}")
print(f"  Total Parameters: {clip_params:,} ({clip_params/1e6:.2f}M)")

print(f"\n✓ U-Net initialized")
print(f"  Parameters: {unet_params:,} ({unet_params/1e6:.2f}M)")
print(f"  ⭐ TARGET ACHIEVED: ~86M parameters!")

print(f"\n📊 Total Parameters (all models): {(vae_params + clip_params + unet_params)/1e6:.2f}M")

print("\n" + "="*80)
print("Proceed to Cell 4 (Training Utilities)")
print("="*80)

In [ ]:

"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    Cell 4: Training Engine & Utilities                       ║
╚══════════════════════════════════════════════════════════════════════════════╝

Complete training engine with:
- Component-specific trainers (VAE, CLIP, U-Net)
- Sampling & visualization
- Checkpointing & logging
- DDIM/DPM-Solver++ samplers
"""

# ═══════════════════════════════════════════════════════════════════════════
# PART 1: DIFFUSION UTILITIES
# ═══════════════════════════════════════════════════════════════════════════

class DDPMScheduler:
    """DDPM noise scheduler"""
    def __init__(self, num_steps: int = 1000, beta_start: float = 0.0001, beta_end: float = 0.02):
        self.num_steps = num_steps

        # Linear beta schedule
        self.betas = torch.linspace(beta_start, beta_end, num_steps)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

        # Calculations for diffusion q(x_t | x_{t-1})
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

        # Calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_log_variance = torch.log(torch.clamp(self.posterior_variance, min=1e-20))
        self.posterior_mean_coef1 = self.betas * torch.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.posterior_mean_coef2 = (1.0 - self.alphas_cumprod_prev) * torch.sqrt(self.alphas) / (1.0 - self.alphas_cumprod)

    def add_noise(self, x_start, noise, timesteps):
        """Forward diffusion: q(x_t | x_0)"""
        sqrt_alpha_cumprod = self.sqrt_alphas_cumprod[timesteps]
        sqrt_one_minus_alpha_cumprod = self.sqrt_one_minus_alphas_cumprod[timesteps]

        # Reshape for broadcasting
        while len(sqrt_alpha_cumprod.shape) < len(x_start.shape):
            sqrt_alpha_cumprod = sqrt_alpha_cumprod.unsqueeze(-1)
            sqrt_one_minus_alpha_cumprod = sqrt_one_minus_alpha_cumprod.unsqueeze(-1)

        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise

    def sample_prev_timestep(self, model_output, timestep, sample):
        """Reverse diffusion: p(x_{t-1} | x_t)"""
        # Get parameters
        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod_prev[timestep]
        beta_prod_t = 1 - alpha_prod_t

        # Predict x_0
        pred_original_sample = (sample - torch.sqrt(beta_prod_t) * model_output) / torch.sqrt(alpha_prod_t)

        # Clip
        pred_original_sample = torch.clamp(pred_original_sample, -1, 1)

        # Compute mean
        pred_sample_direction = torch.sqrt(1 - alpha_prod_t_prev) * model_output
        prev_sample_mean = torch.sqrt(alpha_prod_t_prev) * pred_original_sample + pred_sample_direction

        # Add noise (except for t=0)
        variance = 0
        if timestep > 0:
            noise = torch.randn_like(model_output)
            variance = torch.sqrt(self.posterior_variance[timestep]) * noise

        prev_sample = prev_sample_mean + variance

        return prev_sample

class DDIMScheduler:
    """DDIM sampler - faster sampling"""
    def __init__(self, num_train_steps: int = 1000, num_inference_steps: int = 50):
        self.num_train_steps = num_train_steps
        self.num_inference_steps = num_inference_steps

        # Create timestep schedule
        self.timesteps = torch.linspace(num_train_steps - 1, 0, num_inference_steps, dtype=torch.long)

        # Beta schedule
        beta_start, beta_end = 0.0001, 0.02
        betas = torch.linspace(beta_start, beta_end, num_train_steps)
        alphas = 1.0 - betas
        self.alphas_cumprod = torch.cumprod(alphas, dim=0)

    def sample_prev_timestep(self, model_output, timestep, sample, eta=0.0):
        """DDIM sampling step"""
        prev_timestep = timestep - self.num_train_steps // self.num_inference_steps

        alpha_prod_t = self.alphas_cumprod[timestep]
        alpha_prod_t_prev = self.alphas_cumprod[prev_timestep] if prev_timestep >= 0 else torch.tensor(1.0)

        beta_prod_t = 1 - alpha_prod_t

        # Predict x_0
        pred_original_sample = (sample - torch.sqrt(beta_prod_t) * model_output) / torch.sqrt(alpha_prod_t)
        pred_original_sample = torch.clamp(pred_original_sample, -1, 1)

        # Direction pointing to x_t
        pred_sample_direction = torch.sqrt(1 - alpha_prod_t_prev) * model_output

        prev_sample = torch.sqrt(alpha_prod_t_prev) * pred_original_sample + pred_sample_direction

        return prev_sample

# ═══════════════════════════════════════════════════════════════════════════
# PART 2: COMPONENT TRAINERS
# ═══════════════════════════════════════════════════════════════════════════

class VAETrainer:
    """Trainer for VAE"""
    def __init__(self, model, config, device):
        self.model = model
        self.config = config
        self.device = device
        self.logger = Logger(LOGS_DIR, "vae_training")

        # Optimizer
        if config.use_8bit_adam:
            self.optimizer = bnb.optim.AdamW8bit(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        else:
            self.optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        # Scheduler
        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=100000  # Will be updated
        )

        # AMP
        self.scaler = GradScaler() if config.use_amp else None

        # EMA
        self.ema = EMA(model, config.ema_decay) if config.use_ema else None

        self.global_step = 0
        self.epoch = 0

    def compute_loss(self, recon, target, mu, logvar, kl_weight=1.0):
        """VAE loss with Free Bits to prevent posterior collapse"""

        # Reconstruction loss (MSE)
        recon_loss = F.mse_loss(recon, target, reduction='mean')

        # KL divergence per sample, per latent dimension
        # Shape: [batch_size, latent_channels, height, width]
        kl_per_latent = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())

        # Free bits: Only penalize KL above a threshold
        # This prevents the model from collapsing the latent space
        free_bits_threshold = self.model.free_bits / np.prod(kl_per_latent.shape[1:])
        kl_per_latent = torch.clamp(kl_per_latent, min=free_bits_threshold)

        # Average over batch
        kl_loss = kl_per_latent.mean()

        # Total loss with annealed KL weight
        total_loss = recon_loss + kl_weight * kl_loss

        return total_loss, recon_loss, kl_loss

    def get_kl_weight(self, step, mode='cyclical'):
        """
        KL weight annealing schedule

        Args:
            step: Current training step
            mode: 'cyclical', 'linear', or 'constant'
        """
        if mode == 'constant':
            return self.config.beta

        elif mode == 'linear':
            # Linear warmup from 0 to beta over warmup_steps
            warmup_steps = 10000
            return min(1.0, step / warmup_steps) * self.config.beta

        elif mode == 'cyclical':
            # Cyclical annealing (best for preventing collapse)
            cycle_length = 5000  # steps per cycle
            cycle_progress = (step % cycle_length) / cycle_length

            # Ramp up from 0 to beta over first 50% of cycle
            if cycle_progress < 0.5:
                kl_weight = (cycle_progress * 2) * self.config.beta
            else:
                kl_weight = self.config.beta

            return kl_weight

        return self.config.beta

    def train_step(self, batch):
        """Single training step with KL annealing"""
        self.model.train()
        images = batch["image"].to(self.device)

        # Get annealed KL weight
        kl_weight = self.get_kl_weight(self.global_step, mode='cyclical')

        # Forward pass
        if self.config.use_amp:
            with autocast():
                recon, mu, logvar = self.model(images)
                loss, recon_loss, kl_loss = self.compute_loss(
                    recon, images, mu, logvar, kl_weight=kl_weight
                )
        else:
            recon, mu, logvar = self.model(images)
            loss, recon_loss, kl_loss = self.compute_loss(
                recon, images, mu, logvar, kl_weight=kl_weight
            )

        # Backward pass
        self.optimizer.zero_grad()

        if self.config.use_amp:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
            self.optimizer.step()

        self.scheduler.step()

        if self.ema:
            self.ema.update()

        self.global_step += 1

        return {
            "loss": loss.item(),
            "recon_loss": recon_loss.item(),
            "kl_loss": kl_loss.item(),
            "kl_weight": kl_weight,  # Log this!
            "lr": self.optimizer.param_groups[0]['lr']
        }

    @torch.no_grad()
    def sample(self, num_samples=4):
        """Generate reconstruction samples"""
        self.model.eval()

        # Get validation batch
        val_loader = get_dataloader("vae", batch_size=num_samples)[1]
        batch = next(iter(val_loader))
        images = batch["image"].to(self.device)

        # Apply EMA if available
        if self.ema:
            self.ema.apply_shadow()

        # Reconstruct
        recon, _, _ = self.model(images)

        # Restore weights
        if self.ema:
            self.ema.restore()

        # Denormalize
        images = (images + 1) / 2
        recon = (recon + 1) / 2

        # Create grid
        comparison = torch.cat([images, recon], dim=0)
        grid = make_grid(comparison, nrow=num_samples)

        return grid

class CLIPTrainer:
    """Trainer for CLIP"""
    def __init__(self, model, tokenizer, config, device):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        self.logger = Logger(LOGS_DIR, "clip_training")

        # Optimizer
        if config.use_8bit_adam:
            self.optimizer = bnb.optim.AdamW8bit(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        else:
            self.optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=100000
        )

        self.scaler = GradScaler() if config.use_amp else None
        self.ema = EMA(model, config.ema_decay) if config.use_ema else None

        self.global_step = 0
        self.epoch = 0

    def compute_loss(self, logits_per_image, logits_per_text):
        """Contrastive loss"""
        batch_size = logits_per_image.shape[0]
        labels = torch.arange(batch_size, device=self.device)

        loss_i = F.cross_entropy(logits_per_image, labels)
        loss_t = F.cross_entropy(logits_per_text, labels)

        loss = (loss_i + loss_t) / 2

        # Accuracy
        acc_i = (logits_per_image.argmax(dim=1) == labels).float().mean()
        acc_t = (logits_per_text.argmax(dim=1) == labels).float().mean()

        return loss, acc_i, acc_t

    def train_step(self, batch):
        """Single training step"""
        self.model.train()

        images = batch["image"].to(self.device)
        captions = batch["caption"]

        # Tokenize captions
        text_ids = torch.stack([self.tokenizer.encode(cap) for cap in captions]).to(self.device)

        # Forward pass
        if self.config.use_amp:
            with autocast():
                logits_per_image, logits_per_text = self.model(images, text_ids)
                loss, acc_i, acc_t = self.compute_loss(logits_per_image, logits_per_text)
        else:
            logits_per_image, logits_per_text = self.model(images, text_ids)
            loss, acc_i, acc_t = self.compute_loss(logits_per_image, logits_per_text)

        # Backward pass
        self.optimizer.zero_grad()

        if self.config.use_amp:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
            self.optimizer.step()

        self.scheduler.step()

        if self.ema:
            self.ema.update()

        self.global_step += 1

        return {
            "loss": loss.item(),
            "acc_image": acc_i.item(),
            "acc_text": acc_t.item(),
            "lr": self.optimizer.param_groups[0]['lr']
        }

class UNetTrainer:
    """Trainer for U-Net (Diffusion Model)"""
    def __init__(self, unet, vae, clip, tokenizer, config, device):
        self.unet = unet
        self.model = unet
        self.vae = vae
        self.clip = clip
        self.tokenizer = tokenizer
        self.config = config
        self.device = device
        self.logger = Logger(LOGS_DIR, "unet_training")

        # Freeze VAE and CLIP
        self.vae.eval()
        self.clip.eval()
        for param in self.vae.parameters():
            param.requires_grad = False
        for param in self.clip.parameters():
            param.requires_grad = False

        # Scheduler
        self.noise_scheduler = DDPMScheduler(config.num_diffusion_steps)

        # Move ALL scheduler tensors to device
        scheduler_tensors = [
            'betas', 'alphas', 'alphas_cumprod', 'alphas_cumprod_prev',
            'sqrt_alphas_cumprod', 'sqrt_one_minus_alphas_cumprod',
            'posterior_variance', 'posterior_log_variance',
            'posterior_mean_coef1', 'posterior_mean_coef2'
        ]

        for tensor_name in scheduler_tensors:
            if hasattr(self.noise_scheduler, tensor_name):
                tensor = getattr(self.noise_scheduler, tensor_name)
                setattr(self.noise_scheduler, tensor_name, tensor.to(device))

        print(f"✓ Noise scheduler moved to {device}")

        # Optimizer
        if config.use_8bit_adam:
            self.optimizer = bnb.optim.AdamW8bit(
                unet.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )
        else:
            self.optimizer = torch.optim.AdamW(
                unet.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        self.scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=config.warmup_steps,
            num_training_steps=100000
        )

        self.scaler = GradScaler() if config.use_amp else None
        self.ema = EMA(unet, config.ema_decay) if config.use_ema else None

        self.global_step = 0
        self.epoch = 0

    @torch.no_grad()
    def encode_images(self, images):
        """Encode images to latent space"""
        z, _, _ = self.vae.encode(images)
        return z

    @torch.no_grad()
    def encode_text(self, captions):
        """Encode text to embeddings"""
        text_ids = torch.stack([self.tokenizer.encode(cap) for cap in captions]).to(self.device)
        text_features = self.clip.text_encoder(text_ids)
        # Expand dims for cross-attention: (B, D) -> (B, 1, D)
        text_features = text_features.unsqueeze(1)
        return text_features

    def train_step(self, batch):
        """Single training step"""
        self.unet.train()

        images = batch["image"].to(self.device)
        captions = batch["caption"]
        batch_size = images.shape[0]

        # Encode images to latent space
        with torch.no_grad():
            latents = self.encode_images(images)

            # Classifier-free guidance: randomly drop text conditioning
            if random.random() < self.config.cfg_dropout:
                # Use empty text
                text_embeddings = torch.zeros(batch_size, 1, clip_config.embed_dim).to(self.device)
            else:
                text_embeddings = self.encode_text(captions)

        # Sample noise
        noise = torch.randn_like(latents)

        # Sample timesteps
        timesteps = torch.randint(
            0, self.config.num_diffusion_steps,
            (batch_size,), device=self.device
        ).long()

        # Add noise to latents
        noisy_latents = self.noise_scheduler.add_noise(latents, noise, timesteps)

        # Forward pass
        if self.config.use_amp:
            with autocast():
                noise_pred = self.unet(noisy_latents, timesteps, text_embeddings)
                loss = F.mse_loss(noise_pred, noise)
        else:
            noise_pred = self.unet(noisy_latents, timesteps, text_embeddings)
            loss = F.mse_loss(noise_pred, noise)

        # Backward pass
        self.optimizer.zero_grad()

        if self.config.use_amp:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.unet.parameters(), self.config.gradient_clip)
            self.scaler.step(self.optimizer)
            self.scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.unet.parameters(), self.config.gradient_clip)
            self.optimizer.step()

        self.scheduler.step()

        if self.ema:
            self.ema.update()

        self.global_step += 1

        return {
            "loss": loss.item(),
            "lr": self.optimizer.param_groups[0]['lr']
        }

    @torch.no_grad()
    def sample(self, prompts, num_inference_steps=50, guidance_scale=12.0, use_ddim=True):
        """Generate images from text prompts"""
        self.unet.eval()

        if self.ema:
            self.ema.apply_shadow()

        batch_size = len(prompts)

        # Encode text
        text_embeddings = self.encode_text(prompts)

        # For classifier-free guidance, we need unconditional embeddings
        uncond_embeddings = torch.zeros_like(text_embeddings)

        # Combine embeddings for CFG
        text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

        # Initialize latents
        latents = torch.randn(
            batch_size, unet_config.in_channels,
            unet_config.image_size, unet_config.image_size
        ).to(self.device)

        # Choose scheduler
        if use_ddim:
            scheduler = DDIMScheduler(self.config.num_diffusion_steps, num_inference_steps)
            timesteps = scheduler.timesteps.to(self.device)
        else:
            scheduler = self.noise_scheduler
            timesteps = list(range(self.config.num_diffusion_steps))[::-1]

        # Denoising loop
        for t in tqdm(timesteps, desc="Sampling"):
            # Expand latents for CFG
            latent_model_input = torch.cat([latents] * 2)
            t_input = torch.tensor([t] * (batch_size * 2), device=self.device)

            # Predict noise
            noise_pred = self.unet(latent_model_input, t_input, text_embeddings)

            # Classifier-free guidance
            noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

            # Denoise
            if use_ddim:
                latents = scheduler.sample_prev_timestep(noise_pred, t, latents)
            else:
                latents = scheduler.sample_prev_timestep(noise_pred, t, latents)

        # Decode latents to images
        images = self.vae.decode(latents)
        images = (images + 1) / 2  # Denormalize
        images = torch.clamp(images, 0, 1)

        if self.ema:
            self.ema.restore()

        return images

# ═══════════════════════════════════════════════════════════════════════════
# PART 3: MAIN TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════

def train_component(component: str, config: TrainingConfig):
    """Main training function"""

    print("\n" + "="*80)
    print(f"TRAINING {component.upper()}")
    print("="*80)

    # Get dataloader
    train_loader, val_loader = get_dataloader(component, config.batch_size)

    # Create trainer
    if component == "vae":
        trainer = VAETrainer(vae_model, config, device)

    elif component == "clip":
        trainer = CLIPTrainer(clip_model, tokenizer, config, device)

    elif component == "unet":
      # Load pretrained VAE and CLIP (extract model_state_dict from checkpoint)
      vae_checkpoint = torch.load(config.vae_path, map_location=device)
      clip_checkpoint = torch.load(config.clip_path, map_location=device)

      # Handle both checkpoint formats (full checkpoint vs state_dict only)
      if "model_state_dict" in vae_checkpoint:
          vae_model.load_state_dict(vae_checkpoint["model_state_dict"])
      else:
          vae_model.load_state_dict(vae_checkpoint)

      if "model_state_dict" in clip_checkpoint:
          clip_model.load_state_dict(clip_checkpoint["model_state_dict"])
      else:
          clip_model.load_state_dict(clip_checkpoint)

      print(f"✓ Loaded VAE from {config.vae_path}")
      print(f"✓ Loaded CLIP from {config.clip_path}")

      trainer = UNetTrainer(unet_model, vae_model, clip_model, tokenizer, config, device)

    else:
        raise ValueError(f"Unknown component: {component}")

    # Load checkpoint if resuming
    if config.checkpoint_path and os.path.exists(config.checkpoint_path):
        trainer.epoch, trainer.global_step, _ = load_checkpoint(
            config.checkpoint_path,
            trainer.model,
            trainer.optimizer,
            None,
            trainer.ema.model if trainer.ema else None,
            trainer.scaler
        )

    # Training loop
    print(f"\nStarting training...")
    print(f"  Epochs: {config.num_epochs}")
    print(f"  Batch size: {config.batch_size}")
    print(f"  Total steps: {len(train_loader) * config.num_epochs}")
    print(f"  Device: {device}")

    start_time = time.time()
    best_loss = float('inf')

    for epoch in range(trainer.epoch, config.num_epochs):
        trainer.epoch = epoch
        epoch_loss = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

        for batch in pbar:
            metrics = trainer.train_step(batch)
            epoch_loss += metrics["loss"]

            # Update progress bar
            pbar.set_postfix({k: f"{v:.6f}" for k, v in metrics.items()})

            # Logging
            if trainer.global_step % config.log_every == 0:
                trainer.logger.log_metrics(trainer.global_step, metrics)

            # Sampling
            if trainer.global_step % config.sample_every == 0:
                trainer.logger.log(f"Generating samples at step {trainer.global_step}...")

                if component in ["vae"]:
                    sample_grid = trainer.sample(config.num_samples)
                    save_image(
                        sample_grid,
                        os.path.join(SAMPLES_DIR, f"{component}_step_{trainer.global_step:06d}.png")
                    )

                elif component == "unet":
                    # Sample with different prompts
                    test_prompts = [
                        "a dog on a skateboard",
                        "a cat sitting on a table",
                        "a beautiful sunset",
                        "a person playing guitar"
                    ]
                    sample_images = trainer.sample(test_prompts[:config.num_samples], use_ddim=True)
                    save_image(
                        sample_images,
                        os.path.join(SAMPLES_DIR, f"{component}_step_{trainer.global_step:06d}.png"),
                        nrow=2
                    )

            # Checkpointing
            if trainer.global_step % config.save_every == 0:
                checkpoint_path = os.path.join(
                    CHECKPOINTS_DIR,
                    f"{component}_step_{trainer.global_step:06d}.pt"
                )
                save_checkpoint(
                    trainer.model,
                    trainer.optimizer,
                    trainer.scheduler,
                    epoch,
                    trainer.global_step,
                    metrics["loss"],
                    checkpoint_path,
                    trainer.ema.model if trainer.ema else None,
                    trainer.scaler
                )

                # Save best model
                if metrics["loss"] < best_loss:
                    best_loss = metrics["loss"]
                    best_path = os.path.join(CHECKPOINTS_DIR, f"{component}_best.pt")
                    save_checkpoint(
                        trainer.model,
                        trainer.optimizer,
                        trainer.scheduler,
                        epoch,
                        trainer.global_step,
                        metrics["loss"],
                        best_path,
                        trainer.ema.model if trainer.ema else None,
                        trainer.scaler
                    )

        # Epoch summary
        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - start_time

        trainer.logger.log(
            f"Epoch {epoch+1} complete - Avg Loss: {avg_loss:.6f} - Time: {format_time(elapsed)}"
        )

        # Plot metrics
        if component == "vae":
            trainer.logger.plot_metrics(["loss", "recon_loss", "kl_loss"])
        elif component == "clip":
            trainer.logger.plot_metrics(["loss", "acc_image", "acc_text"])
        else:
            trainer.logger.plot_metrics(["loss"])

    # Final save
    final_path = os.path.join(CHECKPOINTS_DIR, f"{component}_final.pt")
    save_checkpoint(
        trainer.model,
        trainer.optimizer,
        trainer.scheduler,
        config.num_epochs,
        trainer.global_step,
        avg_loss,
        final_path,
        trainer.ema.model if trainer.ema else None,
        trainer.scaler
    )

    # Save EMA model separately
    if trainer.ema:
        ema_path = os.path.join(CHECKPOINTS_DIR, f"{component}_ema.pt")
        torch.save(trainer.ema.shadow, ema_path)
        trainer.logger.log(f"✓ EMA weights saved: {ema_path}")

    total_time = time.time() - start_time
    trainer.logger.log(f"\n{'='*80}")
    trainer.logger.log(f"TRAINING COMPLETE!")
    trainer.logger.log(f"Total time: {format_time(total_time)}")
    trainer.logger.log(f"Final loss: {avg_loss:.6f}")
    trainer.logger.log(f"Best loss: {best_loss:.6f}")
    trainer.logger.log(f"{'='*80}\n")

print("\n" + "="*80)
print("TRAINING ENGINE READY")
print("="*80)
print("\nProceed to Cell 5 to start training!")
print("\nAvailable components:")
print("  1. 'vae' - Train VAE (128x128 -> 16x16)")
print("  2. 'clip' - Train CLIP (Text-Image alignment)")
print("  3. 'unet' - Train U-Net (Diffusion model)")
print("\nExample: train_config.component = 'vae'")
print("         train_component('vae', train_config)")

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                  Cell 5: Unified Configuration & Training                    ║
╚══════════════════════════════════════════════════════════════════════════════╝

A single, unified configuration and training entry point.
Select your target component in the Config class and run train().
"""

import glob
import os

# ============================================================================
# 1. MASTER CONFIGURATION (Control Everything Here)
# ============================================================================

class Config:
    """Master configuration for Toy Stable Diffusion"""

    # --- WHAT TO TRAIN ---
    TARGET = "unet"

    # --- PATHS ---
    PROJECT_ROOT = "/content/drive/MyDrive/ToyStableDiffusion"
    CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")
    LOG_DIR = os.path.join(PROJECT_ROOT, "logs")
    SAMPLE_DIR = os.path.join(PROJECT_ROOT, "samples")

    # --- TRAINING HYPERPARAMETERS ---
    BATCH_SIZE = 64
    NUM_EPOCHS = 100

    # Transformers like CLIP prefer this LR
    LEARNING_RATE = 1e-6

    # Regularization is important for CLIP to generalize
    WEIGHT_DECAY = 0.1
    WARMUP_STEPS = 2000      # Slightly longer warmup for stability

    # --- OPTIMIZATION ---
    USE_8BIT_ADAM = True
    USE_AMP = True
    USE_EMA = True
    GRAD_CLIP = 1.0          # CLIP is naturally more stable than VAE

    # --- LOGGING ---
    SAVE_EVERY = 2000        # Save less frequently (CLIP files are smaller)
    SAMPLE_EVERY = 2000      # CLIP metrics (accuracy) are logged every step anyway
    LOG_EVERY = 100

    # --- ARCHITECTURE OVERRIDES ---
    # (Keep these consistent with your VAE run)
    VAE_LATENT_DIM = 16
    CLIP_EMBED_DIM = 512
    UNET_MODEL_CHANNELS = 192

    @classmethod
    def apply_to_globals(cls):
        # ... (Keep the rest of this method exactly as it was) ...
        # (Paste the standard apply_to_globals logic here)

        for d in [cls.CHECKPOINT_DIR, cls.LOG_DIR, cls.SAMPLE_DIR]:
            os.makedirs(d, exist_ok=True)

        train_config.component = cls.TARGET
        train_config.batch_size = cls.BATCH_SIZE
        train_config.num_epochs = cls.NUM_EPOCHS
        train_config.learning_rate = cls.LEARNING_RATE
        train_config.use_8bit_adam = cls.USE_8BIT_ADAM
        train_config.use_amp = cls.USE_AMP
        train_config.use_ema = cls.USE_EMA
        train_config.gradient_clip = cls.GRAD_CLIP
        train_config.save_every = cls.SAVE_EVERY
        train_config.sample_every = cls.SAMPLE_EVERY
        train_config.log_every = cls.LOG_EVERY

        vae_config.latent_size = cls.VAE_LATENT_DIM
        clip_config.embed_dim = cls.CLIP_EMBED_DIM
        unet_config.model_channels = cls.UNET_MODEL_CHANNELS
        unet_config.context_dim = cls.CLIP_EMBED_DIM

        if cls.TARGET == "unet":
            train_config.vae_path = os.path.join(cls.CHECKPOINT_DIR, "vae_best.pt")
            train_config.clip_path = os.path.join(cls.CHECKPOINT_DIR, "clip_best.pt")

Config.apply_to_globals()

# ============================================================================
# 2. UNIFIED TRAINING FUNCTION
# ============================================================================

def find_latest_checkpoint(component):
    """Automatically finds the checkpoint with the highest step number"""
    pattern = os.path.join(Config.CHECKPOINT_DIR, f"{component}_step_*.pt")
    checkpoints = glob.glob(pattern)

    if not checkpoints:
        return None

    # Sort by step number
    try:
        latest_ckpt = max(checkpoints, key=lambda p: int(p.split("_step_")[-1].split(".")[0]))
        return latest_ckpt
    except ValueError:
        return None

def train():
    """Main execution function (MANUAL OVERRIDE VERSION)"""
    print("="*60)
    print(f"🚀 STARTING TRAINING: {Config.TARGET.upper()}")
    print("="*60)

    # --- ⚠️ FORCE RESUME FROM THE FIXED CHECKPOINT ⚠️ ---
    # We bypass the auto-search because it hates the "_FIXED" filename.
    forced_checkpoint = "/content/drive/MyDrive/ToyStableDiffusion/checkpoints/unet_step_056000_FIXED.pt"

    if os.path.exists(forced_checkpoint):
        print(f"\n🔄 MANUAL RESUME: Loading Fixed Checkpoint!")
        print(f"   Loading: {os.path.basename(forced_checkpoint)}")
        train_config.checkpoint_path = forced_checkpoint
    else:
        # Fallback to auto-search if the fixed file is missing
        print(f"\n⚠️ Fixed checkpoint not found at: {forced_checkpoint}")
        latest_ckpt = find_latest_checkpoint(Config.TARGET)
        if latest_ckpt:
            print(f"🔄 Auto-resume: Found {os.path.basename(latest_ckpt)}")
            train_config.checkpoint_path = latest_ckpt
        else:
            print("\n🆕 No checkpoints found. Starting from scratch (Step 0).")
            train_config.checkpoint_path = None

    # Run Training
    try:
        train_component(Config.TARGET, train_config)
        print(f"\n🎉 {Config.TARGET.upper()} training finished successfully!")
    except KeyboardInterrupt:
        print(f"\n⚠️ Training interrupted by user.")
    except Exception as e:
        print(f"\n❌ Training failed: {e}")
        import traceback
        traceback.print_exc()

# Execute
train()

In [ ]:
"""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    Cell 5.5: VAE Diagnostic Suite                         ║
╚═══════════════════════════════════════════════════════════════════════════╝

This cell runs multiple tests to pinpoint NaN sources.
"""

import torch
import torch.nn.functional as F

print("="*80)
print("🔍 COMPREHENSIVE VAE DIAGNOSTICS")
print("="*80)

# ============================================================================
# TEST 1: Check if the checkpoint itself is corrupted
# ============================================================================
print("\n📋 TEST 1: Checkpoint Integrity")
print("-" * 80)
real_checkpoint_path = '/content/drive/MyDrive/ToyStableDiffusion/checkpoints/vae_best.pt'

checkpoint_path = real_checkpoint_path
try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    vae_model.load_state_dict(checkpoint['model_state_dict'])

    # Check for NaN in weights
    nan_params = []
    for name, param in vae_model.named_parameters():
        if torch.isnan(param).any():
            nan_params.append(name)

    if nan_params:
        print(f"❌ CRITICAL: Checkpoint has NaN weights in:")
        for name in nan_params:
            print(f"   - {name}")
        print("\n⚠️  You must use an earlier checkpoint!")
        CHECKPOINT_CORRUPTED = True
    else:
        print("✅ Checkpoint weights are clean (no NaN)")
        CHECKPOINT_CORRUPTED = False

except Exception as e:
    print(f"❌ Failed to load checkpoint: {e}")
    CHECKPOINT_CORRUPTED = True

if CHECKPOINT_CORRUPTED:
    print("\n🔄 Attempting to load previous checkpoint...")
    checkpoint_path = real_checkpoint_path
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        vae_model.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ Loaded {checkpoint_path}")
    else:
        print("❌ No fallback checkpoint found!")

# ============================================================================
# TEST 2: Forward Pass (Inference Mode)
# ============================================================================
print("\n📋 TEST 2: Forward Pass (No Gradients)")
print("-" * 80)

vae_model.eval()
test_loader, _ = get_dataloader("vae", batch_size=8)
test_batch = next(iter(test_loader))
images = test_batch["image"].to(device)

with torch.no_grad():
    try:
        recon, mu, logvar = vae_model(images)

        print(f"Input  → range: [{images.min():.4f}, {images.max():.4f}], mean: {images.mean():.4f}")
        print(f"Recon  → range: [{recon.min():.4f}, {recon.max():.4f}], mean: {recon.mean():.4f}")
        print(f"Mu     → range: [{mu.min():.4f}, {mu.max():.4f}], mean: {mu.mean():.4f}, std: {mu.std():.4f}")
        print(f"Logvar → range: [{logvar.min():.4f}, {logvar.max():.4f}], mean: {logvar.mean():.4f}")

        # Check for NaN/Inf
        checks = {
            "Reconstruction": recon,
            "Mu": mu,
            "Logvar": logvar
        }

        has_issues = False
        for name, tensor in checks.items():
            nan_count = torch.isnan(tensor).sum().item()
            inf_count = torch.isinf(tensor).sum().item()

            if nan_count > 0 or inf_count > 0:
                print(f"❌ {name}: {nan_count} NaN, {inf_count} Inf")
                has_issues = True

        if not has_issues:
            print("✅ Forward pass clean (no NaN/Inf)")

    except Exception as e:
        print(f"❌ Forward pass failed: {e}")
        import traceback
        traceback.print_exc()

# ============================================================================
# TEST 3: Loss Computation
# ============================================================================
print("\n📋 TEST 3: Loss Computation")
print("-" * 80)

with torch.no_grad():
    try:
        # Reconstruction loss
        recon_loss = F.mse_loss(recon, images, reduction='mean')
        print(f"Recon Loss: {recon_loss.item():.6f}")

        # KL divergence (manual computation to check each step)
        kl_elementwise = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
        print(f"KL (before sum): min={kl_elementwise.min():.6f}, max={kl_elementwise.max():.6f}, mean={kl_elementwise.mean():.6f}")

        kl_loss = kl_elementwise.sum() / mu.shape[0]
        print(f"KL Loss: {kl_loss.item():.6f}")

        # Check for problematic values in KL computation
        exp_logvar = logvar.exp()
        print(f"exp(logvar): min={exp_logvar.min():.6f}, max={exp_logvar.max():.6f}")

        mu_squared = mu.pow(2)
        print(f"mu^2: min={mu_squared.min():.6f}, max={mu_squared.max():.6f}")

        if torch.isnan(kl_loss) or torch.isinf(kl_loss):
            print("❌ KL Loss is NaN/Inf!")
        else:
            print("✅ Loss computation clean")

    except Exception as e:
        print(f"❌ Loss computation failed: {e}")

# ============================================================================
# TEST 4: Backward Pass (Single Gradient Step)
# ============================================================================
print("\n📋 TEST 4: Single Training Step Simulation")
print("-" * 80)

vae_model.train()
optimizer_test = torch.optim.AdamW(vae_model.parameters(), lr=1e-5)

try:
    # Forward
    recon, mu, logvar = vae_model(images)

    # Loss
    recon_loss = F.mse_loss(recon, images, reduction='mean')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.shape[0]
    loss = recon_loss + vae_config.beta * kl_loss

    print(f"Loss before backward: {loss.item():.6f}")

    # Backward
    optimizer_test.zero_grad()
    loss.backward()

    # Check gradients
    max_grad = 0.0
    nan_grad_params = []

    for name, param in vae_model.named_parameters():
        if param.grad is not None:
            grad_norm = param.grad.norm().item()
            max_grad = max(max_grad, grad_norm)

            if torch.isnan(param.grad).any():
                nan_grad_params.append(name)

    if nan_grad_params:
        print(f"❌ NaN gradients detected in:")
        for name in nan_grad_params[:5]:  # Show first 5
            print(f"   - {name}")
    else:
        print(f"✅ Gradients clean. Max gradient norm: {max_grad:.6f}")

        if max_grad > 100:
            print(f"⚠️  WARNING: Very large gradients ({max_grad:.2f})! Gradient clipping recommended.")

except Exception as e:
    print(f"❌ Backward pass failed: {e}")
    import traceback
    traceback.print_exc()

# ============================================================================
# TEST 5: Encoder/Decoder Stability
# ============================================================================
print("\n📋 TEST 5: Component-Level Check")
print("-" * 80)

vae_model.eval()
with torch.no_grad():
    try:
        # Test encoder
        mu, logvar = vae_model.encoder(images)
        print(f"Encoder output:")
        print(f"  Mu: range=[{mu.min():.4f}, {mu.max():.4f}], has_nan={torch.isnan(mu).any()}")
        print(f"  Logvar: range=[{logvar.min():.4f}, {logvar.max():.4f}], has_nan={torch.isnan(logvar).any()}")

        # Test reparameterization
        z = vae_model.reparameterize(mu, logvar)
        print(f"Latent z: range=[{z.min():.4f}, {z.max():.4f}], has_nan={torch.isnan(z).any()}")

        # Test decoder
        recon = vae_model.decoder(z)
        print(f"Decoder output: range=[{recon.min():.4f}, {recon.max():.4f}], has_nan={torch.isnan(recon).any()}")

        print("✅ All components functional")

    except Exception as e:
        print(f"❌ Component test failed: {e}")

# ============================================================================
# TEST 6: Distribution Statistics
# ============================================================================
print("\n📋 TEST 6: Latent Space Distribution")
print("-" * 80)

with torch.no_grad():
    all_mus = []
    all_logvars = []

    for i, batch in enumerate(test_loader):
        if i >= 5:  # Test on 5 batches
            break
        imgs = batch["image"].to(device)
        mu, logvar = vae_model.encoder(imgs)
        all_mus.append(mu)
        all_logvars.append(logvar)

    all_mus = torch.cat(all_mus, dim=0)
    all_logvars = torch.cat(all_logvars, dim=0)

    print(f"Mu statistics (across {all_mus.shape[0]} samples):")
    print(f"  Mean: {all_mus.mean():.4f}, Std: {all_mus.std():.4f}")
    print(f"  Min: {all_mus.min():.4f}, Max: {all_mus.max():.4f}")
    print(f"  % values > 5: {(all_mus.abs() > 5).float().mean().item()*100:.2f}%")

    print(f"\nLogvar statistics:")
    print(f"  Mean: {all_logvars.mean():.4f}, Std: {all_logvars.std():.4f}")
    print(f"  Min: {all_logvars.min():.4f}, Max: {all_logvars.max():.4f}")
    print(f"  % values > 5: {(all_logvars.abs() > 5).float().mean().item()*100:.2f}%")

    if all_logvars.max() > 10:
        print("⚠️  WARNING: Logvar values are very high! Risk of exp() overflow.")

    if all_mus.abs().max() > 10:
        print("⚠️  WARNING: Mu values are very extreme!")

# ============================================================================
# SUMMARY & RECOMMENDATIONS
# ============================================================================
print("\n" + "="*80)
print("📊 DIAGNOSTIC SUMMARY")
print("="*80)

recommendations = []

if CHECKPOINT_CORRUPTED:
    recommendations.append("🔴 CRITICAL: Use an earlier checkpoint (vae_step_015000.pt or earlier)")

if 'max_grad' in locals() and max_grad > 100:
    recommendations.append("🟡 Reduce gradient clipping to 0.5 or lower")

if 'all_logvars' in locals() and all_logvars.max() > 10:
    recommendations.append("🟡 Add stricter logvar clamping: torch.clamp(logvar, -10, 10)")

if 'all_mus' in locals() and all_mus.abs().max() > 10:
    recommendations.append("🟡 Add mu clamping: torch.clamp(mu, -10, 10)")

if vae_config.beta > 0.001:
    recommendations.append("🟡 Reduce KL weight (beta) to 0.0001 or lower")

if recommendations:
    print("\n⚠️  RECOMMENDED FIXES:")
    for i, rec in enumerate(recommendations, 1):
        print(f"  {i}. {rec}")
else:
    print("\n✅ No critical issues detected. Model appears stable.")
    print("   NaN likely occurred due to a rare bad batch or accumulated numerical drift.")
    print("   Apply the preventive fixes (clamping, lower LR, stricter grad clip) anyway.")

print("\n" + "="*80)